# Iso-FLOPs Calculator: Looped vs. Baseline Transformer

This notebook computes training FLOPs for:
- **Baseline**: standard transformer (BASELINE_266M)
- **Looped**: shared-weight transformer with L recurrence steps (3/5/7/9/11 MTP)

Then it answers: to match the looped model's FLOPs, how many **tokens** or **layers** does the baseline need?

---
### FLOPs convention
We use the standard **training FLOPs = 6 · N · D** approximation (Hoffmann et al., Chinchilla),  
where N = number of parameters and D = number of training tokens.  
This works because forward ≈ 2·N·D MACs and backward ≈ 2× forward, giving 6·N·D total.

For the looped model the *effective* parameter count is higher because the shared blocks are  
**executed L times per forward pass**, so we compute FLOPs directly per-token instead.

In [55]:
import math
import pandas as pd
pd.set_option('display.float_format', '{:.3e}'.format)

## 1. Model Architecture Parameters
Edit the values below to match your configs.

In [56]:
import math

# ── Shared / common ──────────────────────────────────────────────────────────
n_embd      = 1024    # hidden dimension
n_head      = 32      # number of attention heads
vocab_size  = 50304   # vocabulary size
seq_len     = 2048    # sequence length (for attention FLOPs)

# ── Baseline (BASELINE_266M) ─────────────────────────────────────────────────
n_layer_base    = 12     # number of transformer blocks
ffn_hidden_base = 4096   # ffn_hidden as set in config (SwiGLU will resize internally)
D_base          = 6_848_839_680  # training tokens

# ── Looped model (3MTP config) ───────────────────────────────────────────────
n_layer_loop    = 12     # number of SHARED transformer blocks (executed L times)
ffn_hidden_loop = 4096   # ffn_hidden as set in config
D_loop          = 6_848_839_680  # training tokens
loop_values     = [3, 5, 7, 9, 11]  # L values to evaluate

# ── Looped model: extra components ───────────────────────────────────────────
use_recurrence_embedding        = True   # proj input is 2d+1 (adds a scalar recurrence index)
use_shared_ws_gate              = True   # 1 shared gate Linear(d,d) for all L steps
use_per_iter_norms              = True   # L separate LayerNorm(d+1) for prev_iter_embd
use_per_iter_norms_tokens_embds = False  # single shared LayerNorm(d) for token embeddings

# ── SwiGLU hidden dim correction ─────────────────────────────────────────────
# The SwiGLU implementation uses ceil(2/3 * ffn_hidden, multiple_of=256).
# This is the actual number of hidden units used in W, V, W_2.
# enforce_swiglu_hidden_dim_multiple_of defaults to 256 in the codebase.
SWIGLU_MULTIPLE_OF = 256

def swiglu_actual_hidden(ffn_hidden, multiple_of=SWIGLU_MULTIPLE_OF):
    """Actual SwiGLU hidden dim after the 2/3 scaling + rounding."""
    raw = int(2 * ffn_hidden / 3)
    return math.ceil(raw / multiple_of) * multiple_of

ffn_actual_base = swiglu_actual_hidden(ffn_hidden_base)
ffn_actual_loop = swiglu_actual_hidden(ffn_hidden_loop)
print(f"Baseline FFN: config={ffn_hidden_base} → actual SwiGLU hidden={ffn_actual_base}")
print(f"Looped  FFN: config={ffn_hidden_loop} → actual SwiGLU hidden={ffn_actual_loop}")

Baseline FFN: config=4096 → actual SwiGLU hidden=2816
Looped  FFN: config=4096 → actual SwiGLU hidden=2816


## 2. FLOPs Formulas

For a single transformer block (attention + SwiGLU FFN) processing **T** tokens:

| Component | FLOPs (multiply-adds × 2) |
|---|---|
| QKV projection | `2 × T × 3 × d × d` |
| Attention scores + weighted sum | `2 × T² × d` (causal: `T²/2 × d × 2` → `T² × d`) |
| Output projection | `2 × T × d × d` |
| SwiGLU gate proj | `2 × T × d × h` |
| SwiGLU up proj | `2 × T × d × h` |
| SwiGLU down proj | `2 × T × h × d` |

We count **MACs × 2 = FLOPs** (a multiply-add = 2 FLOPs).

LayerNorms are O(d) per token — negligible vs. linear layers, but we include them for completeness.

Embedding lookup and positional embedding are just table reads — **zero** FLOPs.

LM head: `2 × T × d × V`

In [57]:
def flops_attention(T, d, n_heads):
    """FLOPs for multi-head self-attention (causal) for T tokens, dim d."""
    qkv  = 2 * T * 3 * d * d          # QKV projections
    # Causal attention: each token i attends to i tokens on average → T*(T+1)/2 ≈ T²/2
    attn = 2 * T * T * d              # scores + weighted sum (full, upper triangular masked)
    out  = 2 * T * d * d              # output projection
    return qkv + attn + out

def flops_swiglu_ffn(T, d_model, d_hidden):
    """FLOPs for SwiGLU FFN: gate_proj, up_proj, down_proj (each d_model↔d_hidden)."""
    gate = 2 * T * d_model * d_hidden
    up   = 2 * T * d_model * d_hidden
    down = 2 * T * d_hidden * d_model
    return gate + up + down

def flops_linear(T, d_in, d_out):
    return 2 * T * d_in * d_out

def flops_layernorm(T, d):
    """RMSNorm: ~5 FLOPs per element (mean_sq, rsqrt, scale, multiply, add)."""
    return 5 * T * d

def flops_one_transformer_block(T, d, n_heads, d_ffn):
    return flops_attention(T, d, n_heads) + flops_swiglu_ffn(T, d, d_ffn)

def flops_lm_head(T, d, V):
    return flops_linear(T, d, V)

print("FLOPs helpers defined.")

FLOPs helpers defined.


## 3. Baseline FLOPs

In [58]:
def compute_baseline_forward_flops_per_token(n_layer, d, n_heads, d_ffn_actual, V, T):
    """Forward FLOPs for the baseline. d_ffn_actual is the true SwiGLU hidden dim."""
    block_flops   = flops_one_transformer_block(T, d, n_heads, d_ffn_actual)
    lm_flops      = flops_lm_head(T, d, V)
    total_per_seq = n_layer * block_flops + lm_flops
    return total_per_seq / T  # per token

T = seq_len
base_fwd_per_token = compute_baseline_forward_flops_per_token(
    n_layer_base, n_embd, n_head, ffn_actual_base, vocab_size, T
)
base_train_per_token = 3 * base_fwd_per_token
base_total_flops = base_train_per_token * D_base

print(f"Baseline forward FLOPs/token : {base_fwd_per_token:.4e}")
print(f"Baseline train  FLOPs/token  : {base_train_per_token:.4e}")
print(f"Baseline total train FLOPs   : {base_total_flops:.4e}  ({D_base:.2e} tokens)")

Baseline forward FLOPs/token : 4.6164e+08
Baseline train  FLOPs/token  : 1.3849e+09
Baseline total train FLOPs   : 9.4850e+18  (6.85e+09 tokens)


## 4. Looped Model FLOPs

Per forward pass the looped model does:
1. **Token embedding** (free)
2. For each loop iteration `l = 1..L`:
   - `prev_iter_embd_norm` — LayerNorm on `[prev_h | scalar]`, shape `d+1`
   - `embd_norm` on tokens embedding, shape `d`
   - **proj**: `Linear(2d+1, d)` to fuse previous iteration hidden state + current token embd
   - All **n_layer transformer blocks** (shared weights, but same FLOPs)
   - **gate**: `Linear(d, d)` applied to the output hidden state
   - Element-wise gate multiply + accumulate into combined representation
3. **LM head** once (MTP lambda=0, so only final output used)

In [59]:
def compute_looped_forward_flops_per_token(L, n_layer, d, n_heads, d_ffn_actual, V, T,
                                            use_recurrence_embd=True,
                                            use_shared_ws_gate=True,
                                            use_per_iter_norms=True,
                                            use_per_iter_norms_tokens_embds=False):
    """
    Forward FLOPs for one token in the looped model with L recurrence steps.
    d_ffn_actual is the true SwiGLU hidden dim (after 2/3 scaling + rounding).
    Returns (total_per_seq, breakdown_dict).
    """
    d_proj_in = d * 2 + (1 if use_recurrence_embd else 0)
    d_ln      = d + (1 if use_recurrence_embd else 0)

    per_iter_proj_flops    = flops_linear(T, d_proj_in, d)
    per_iter_gate_flops    = flops_linear(T, d, d)
    per_iter_ln_prev_flops = flops_layernorm(T, d_ln)
    per_iter_ln_embd_flops = flops_layernorm(T, d)

    # transformer blocks run n_layer times per step; proj/gate/LNs run once per step
    per_iter_flops = (
        n_layer * flops_one_transformer_block(T, d, n_heads, d_ffn_actual)
        + per_iter_proj_flops
        + per_iter_gate_flops
        + per_iter_ln_prev_flops
        + per_iter_ln_embd_flops
    )

    loop_total    = L * per_iter_flops
    lm_flops      = flops_lm_head(T, d, V)
    total_per_seq = loop_total + lm_flops

    breakdown = {
        "transformer_blocks": L * n_layer * flops_one_transformer_block(T, d, n_heads, d_ffn_actual),
        "proj (fuse)":        L * per_iter_proj_flops,
        "gate (WS)":          L * per_iter_gate_flops,
        "ln_prev_iter":       L * per_iter_ln_prev_flops,
        "ln_embd":            L * per_iter_ln_embd_flops,
        "lm_head":            lm_flops,
    }
    return total_per_seq, breakdown


print("Looped FLOPs per token breakdown (T=seq_len, forward only):\n")
rows = []
for L in loop_values:
    total_fwd, bd = compute_looped_forward_flops_per_token(
        L, n_layer_loop, n_embd, n_head, ffn_actual_loop, vocab_size, seq_len,
        use_recurrence_embd=use_recurrence_embedding,
        use_shared_ws_gate=use_shared_ws_gate,
        use_per_iter_norms=use_per_iter_norms,
        use_per_iter_norms_tokens_embds=use_per_iter_norms_tokens_embds,
    )
    total_train = 3 * total_fwd / seq_len
    rows.append({"L": L,
                 "fwd FLOPs/token":   total_fwd / seq_len,
                 "train FLOPs/token": total_train,
                 "total train FLOPs": total_train * D_loop,
                 **{k: v / seq_len for k, v in bd.items()}})

df = pd.DataFrame(rows).set_index("L")
print(df[["fwd FLOPs/token", "train FLOPs/token", "total train FLOPs"]].to_string())

Looped FLOPs per token breakdown (T=seq_len, forward only):

    fwd FLOPs/token  train FLOPs/token  total train FLOPs
L                                                        
3         1.198e+09          3.593e+09          2.461e+19
5         1.928e+09          5.783e+09          3.961e+19
7         2.657e+09          7.972e+09          5.460e+19
9         3.387e+09          1.016e+10          6.960e+19
11        4.117e+09          1.235e+10          8.459e+19


## 5. Comparison: Looped vs. Baseline

In [60]:
print(f"Baseline total train FLOPs : {base_total_flops:.4e}\n")
print(f"{'L':>4}  {'Loop total FLOPs':>18}  {'Ratio (loop/base)':>18}")
print("-" * 46)
for row in rows:
    L = row["L"]
    tf = row["total train FLOPs"]
    ratio = tf / base_total_flops
    print(f"{L:>4}  {tf:>18.4e}  {ratio:>18.3f}x")

Baseline total train FLOPs : 9.4850e+18

   L    Loop total FLOPs   Ratio (loop/base)
----------------------------------------------
   3          2.4610e+19               2.595x
   5          3.9606e+19               4.176x
   7          5.4601e+19               5.757x
   9          6.9597e+19               7.338x
  11          8.4592e+19               8.919x


## 6. Iso-FLOPs: Match by Training Tokens

To run the **baseline** for the same total FLOPs as a looped model with L steps:

$$D_{\text{iso}} = \frac{\text{FLOPs}_{\text{loop}}}{3 \times \text{FLOPs/token}_{\text{base}}}$$

In [61]:
print("Tokens baseline must train on to match looped model FLOPs:\n")
print(f"{'L':>4}  {'Loop FLOPs':>14}  {'Iso-FLOPs tokens':>18}  {'Multiplier vs D_base':>22}")
print("-" * 64)
for row in rows:
    L = row["L"]
    loop_flops = row["total train FLOPs"]
    iso_tokens = loop_flops / base_train_per_token
    mult = iso_tokens / D_base
    print(f"{L:>4}  {loop_flops:>14.4e}  {iso_tokens:>18.4e}  {mult:>22.2f}x")

Tokens baseline must train on to match looped model FLOPs:

   L      Loop FLOPs    Iso-FLOPs tokens    Multiplier vs D_base
----------------------------------------------------------------
   3      2.4610e+19          1.7770e+10                    2.59x
   5      3.9606e+19          2.8598e+10                    4.18x
   7      5.4601e+19          3.9426e+10                    5.76x
   9      6.9597e+19          5.0254e+10                    7.34x
  11      8.4592e+19          6.1082e+10                    8.92x


## 7. Iso-FLOPs: Match by Adding Layers to Baseline

Keep the same `D_base` tokens. How many layers does the baseline need so its total FLOPs equals the looped model?

$$n_{\text{layer,iso}} = \frac{\text{FLOPs}_{\text{loop}} / (3 \cdot D_{\text{base}}) - F_{\text{lm\_head}}}{F_{\text{block}}}$$

In [62]:
T = seq_len
flops_per_block_per_token = flops_one_transformer_block(T, n_embd, n_head, ffn_actual_base) / T
flops_lmhead_per_token    = flops_lm_head(T, n_embd, vocab_size) / T

print("Layers baseline needs (same D_base tokens) to match looped model FLOPs:\n")
print(f"{'L':>4}  {'Loop FLOPs':>14}  {'Iso layers (float)':>20}  {'Iso layers (ceil)':>18}")
print("-" * 62)
for row in rows:
    L = row["L"]
    loop_flops = row["total train FLOPs"]
    target_fwd_per_token = loop_flops / (3 * D_base)
    iso_layers = (target_fwd_per_token - flops_lmhead_per_token) / flops_per_block_per_token
    print(f"{L:>4}  {loop_flops:>14.4e}  {iso_layers:>20.2f}  {math.ceil(iso_layers):>18}")

Layers baseline needs (same D_base tokens) to match looped model FLOPs:

   L      Loop FLOPs    Iso layers (float)   Iso layers (ceil)
--------------------------------------------------------------
   3      2.4610e+19                 36.63                  37
   5      3.9606e+19                 61.05                  62
   7      5.4601e+19                 85.48                  86
   9      6.9597e+19                109.90                 110
  11      8.4592e+19                134.32                 135


## 8. Parameter Counts (for reference)

Parameter counts are independent of FLOPs but useful for the iso-param comparison you already did.

In [63]:
def count_params_one_block(d, d_ffn_actual, head_dim):
    """
    Parameters in one GPT2Block (bias=False throughout).
    - Attention: Q, K, V, O each Linear(d, d), no bias → 4*d*d
    - QK norm: q_norm + k_norm, each RMSNorm(head_dim) → 2*head_dim
    - FFN (SwiGLU): W, V, W_2 each Linear(d, d_ffn_actual) / Linear(d_ffn_actual, d), no bias → 3*d*d_ffn_actual
    - Block norms: attention_norm + ffn_norm, each RMSNorm(d) → 2*d
    """
    attn  = 4 * d * d + 2 * head_dim
    ffn   = 3 * d * d_ffn_actual
    norms = 2 * d
    return attn + ffn + norms

def count_params_baseline(n_layer, d, d_ffn_actual, V, head_dim):
    wte     = V * d
    blocks  = n_layer * count_params_one_block(d, d_ffn_actual, head_dim)
    lm_norm = d        # lm_head_norm RMSNorm(d)
    lm_head = V * d    # use_weight_tying=False
    return wte + blocks + lm_norm + lm_head

def count_params_looped(n_layer, d, d_ffn_actual, V, L, head_dim,
                         use_recurrence_embd=True,
                         use_shared_ws_gate=True,
                         use_per_iter_norms=True,
                         use_per_iter_norms_tokens_embds=False):
    wte     = V * d
    blocks  = n_layer * count_params_one_block(d, d_ffn_actual, head_dim)  # shared weights, counted once
    lm_norm = d
    lm_head = V * d

    # proj: Linear(2d+1, d) with bias (nn.Linear default) → (2d+1)*d + d
    d_proj_in = d * 2 + (1 if use_recurrence_embd else 0)
    proj      = d_proj_in * d + d

    # gate: shared Linear(d, d) with bias → d*d + d
    gate      = (d * d + d) if use_shared_ws_gate else L * (d * d + d)

    # scalar gate_bias: nn.Parameter(L,) — separate from the Linear bias
    gate_bias_param = L

    # prev_iter_embd_norm: LayerNorm(d+1) has weight AND bias → 2*(d+1) each
    # (confirmed: module shows 2050 params per norm for d+1=1025)
    d_ln    = d + (1 if use_recurrence_embd else 0)
    ln_prev = (L * 2 * d_ln) if use_per_iter_norms else (2 * d_ln)

    # embd_norm: LayerNorm(d) with weight AND bias → 2*d
    ln_embd = (L * 2 * d) if use_per_iter_norms_tokens_embds else (2 * d)

    # latent_thoughts: nn.Parameter(L-1, d)
    latent  = (L - 1) * d

    # halt_block: Linear(d, 1) with bias → d + 1
    halt    = d + 1

    return wte + blocks + lm_norm + lm_head + proj + gate + gate_bias_param + ln_prev + ln_embd + latent + halt


head_dim_val = n_embd // n_head

base_params = count_params_baseline(n_layer_base, n_embd, ffn_actual_base, vocab_size, head_dim_val)
print(f"Baseline parameters : {base_params:,}  ({base_params/1e6:.1f}M)\n")
print(f"{'L':>4}  {'Loop params':>14}  {'Loop params (M)':>16}  {'Ratio vs base':>14}")
print("-" * 54)
for L in loop_values:
    lp = count_params_looped(
        n_layer_loop, n_embd, ffn_actual_loop, vocab_size, L, head_dim_val,
        use_recurrence_embd=use_recurrence_embedding,
        use_shared_ws_gate=use_shared_ws_gate,
        use_per_iter_norms=use_per_iter_norms,
        use_per_iter_norms_tokens_embds=use_per_iter_norms_tokens_embds,
    )
    note = "  ← matches W&B (260,368,156)" if L == 3 else ""
    print(f"{L:>4}  {lp:>14,}  {lp/1e6:>16.1f}M  {lp/base_params:>14.3f}x{note}")

Baseline parameters : 257,189,632  (257.2M)

   L     Loop params   Loop params (M)   Ratio vs base
------------------------------------------------------
   3     260,349,706             260.3M           1.012x  ← matches W&B (260,368,156)
   5     260,355,856             260.4M           1.012x
   7     260,362,006             260.4M           1.012x
   9     260,368,156             260.4M           1.012x
  11     260,374,306             260.4M           1.012x


## 9. Full Iso-FLOPs Summary Table

In [64]:
summary = []
for L in loop_values:
    total_fwd, _ = compute_looped_forward_flops_per_token(
        L, n_layer_loop, n_embd, n_head, ffn_hidden_loop, vocab_size, seq_len,
        use_recurrence_embd=use_recurrence_embedding,
        use_shared_ws_gate=use_shared_ws_gate,
        use_per_iter_norms=use_per_iter_norms,
        use_per_iter_norms_tokens_embds=use_per_iter_norms_tokens_embds,
    )
    loop_train_per_tok = 3 * total_fwd / seq_len
    loop_total = loop_train_per_tok * D_loop

    iso_tokens = loop_total / base_train_per_token
    iso_token_mult = iso_tokens / D_base

    target_fwd_per_tok = loop_total / (3 * D_base)
    iso_layers_float = (target_fwd_per_tok - flops_lmhead_per_token) / flops_per_block_per_token

    loop_params = count_params_looped(
        n_layer_loop, n_embd, ffn_hidden_loop, vocab_size, L, head_dim_val,
        use_recurrence_embd=use_recurrence_embedding,
        use_shared_ws_gate=use_shared_ws_gate,
        use_per_iter_norms=use_per_iter_norms,
        use_per_iter_norms_tokens_embds=use_per_iter_norms_tokens_embds,
    )

    summary.append({
        "L (loops)": L,
        "Loop params (M)": round(loop_params / 1e6, 1),
        "Loop total FLOPs": f"{loop_total:.3e}",
        "Flop ratio vs base": round(loop_total / base_total_flops, 2),
        "Iso-tokens (base)": f"{iso_tokens:.3e}",
        "Token mult vs D_base": round(iso_token_mult, 2),
        "Iso-layers (base, same D)": round(iso_layers_float, 1),
        "Iso-layers (ceil)": math.ceil(iso_layers_float),
    })

df_summary = pd.DataFrame(summary).set_index("L (loops)")
print("\n=== ISO-FLOP SUMMARY ===")
print(f"Baseline: {n_layer_base} layers, {ffn_hidden_base} FFN dim, {D_base:.2e} tokens")
print(f"Baseline params: {base_params/1e6:.1f}M  |  Baseline total FLOPs: {base_total_flops:.3e}\n")
print(df_summary.to_string())


=== ISO-FLOP SUMMARY ===
Baseline: 12 layers, 4096 FFN dim, 6.85e+09 tokens
Baseline params: 257.2M  |  Baseline total FLOPs: 9.485e+18

           Loop params (M) Loop total FLOPs  Flop ratio vs base Iso-tokens (base)  Token mult vs D_base  Iso-layers (base, same D)  Iso-layers (ceil)
L (loops)                                                                                                                                            
3                3.075e+02        3.043e+19           3.210e+00         2.197e+10             3.210e+00                  4.610e+01                 47
5                3.075e+02        4.930e+19           5.200e+00         3.560e+10             5.200e+00                  7.680e+01                 77
7                3.075e+02        6.817e+19           7.190e+00         4.923e+10             7.190e+00                  1.076e+02                108
9                3.076e+02        8.705e+19           9.180e+00         6.285e+10             9.180e+00         